# 2. An assembly over its life

This tutorial follows one assembly through its life: construction, two
replacements (module B4) and end of life (C1–C4).
[`timeline()`](../api/inventory.md#dlca.inventory.timeline) places each
emission in its year, and `assess()` turns the yearly series into climate
results.

!!! note "Illustrative numbers"
    The assembly is the steel frame of the example file
    [`tests/data/inventories/steel.json`](https://github.com/jayarehart/DLCA/tree/main/tests/data/inventories):
    steel beams and their intumescent (fire) coating. Every number in that
    file is made up.

In [ ]:
import json
from pathlib import Path

# The example inventories live in the repository's tests/data/inventories/.
# Look for them from the folder this notebook runs in, upwards.
here = Path.cwd().resolve()
DATA = next(
    p / "tests" / "data" / "inventories"
    for p in [here, *here.parents]
    if (p / "tests" / "data" / "inventories").is_dir()
)


def load(name):
    """The example file `name`.json as a plain dict."""
    return json.loads((DATA / f"{name}.json").read_text(encoding="utf-8"))

## Make a one-assembly input

`steel.json` is a whole floor with two assemblies. Here we keep only the
frame, as a [`BuildingInventory`](../api/inventory.md#dlca.inventory.BuildingInventory)
of its own. The coating has two B4 events, in years 25 and 50: each holds the
emissions of the new coating and of removing the old one, entered by hand.

In [ ]:
from dlca.inventory import BuildingInventory

data = load("steel")
data["name"] = "Steel frame only (illustrative)"
data["assemblies"] = [a for a in data["assemblies"] if a["name"] == "Frame"]
frame = BuildingInventory.model_validate(data)

for material in frame.assemblies[0].materials:
    print(
        material.name, material.quantity, material.unit, "B4 events:", [e.year for e in material.b4]
    )

## Place the emissions in time

`timeline()` gives one series per (assembly, material, module). The series
runs from year 0 to the end-of-life year (study period + 1) plus the
horizon, so the last emissions get a full horizon too.

In [ ]:
import pandas as pd

from dlca.inventory import timeline

t = timeline(frame, horizon=100)
print("study period:", t.study_period, "years; end of life in year", t.end_of_life_year)
print("series length:", t.years, "years")

rows = []
for (_, material, module), series in t.parts.items():
    years = [int(y) for y in series["co2"].nonzero()[0]]
    rows.append((material, module, years, series["co2"].sum()))
pd.DataFrame(rows, columns=["material", "module", "years with CO2", "CO2 [kg]"])

A1–A5 fall in year 0, the coating's B4 in years 25 and 50, and C1–C4 in
year 61. The amounts are the input amounts times the quantity; the timing
does not change them.

## Assess the yearly series

`Timeline.emissions()` adds up all the parts and goes straight into
`assess()`. Use the same horizon for both.

In [ ]:
import matplotlib.pyplot as plt

from dlca import assess

result = assess(t.emissions(), parameters="AR6", horizon=t.horizon)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
co2 = t.emissions()["co2"]
ax1.bar(range(t.years), co2 / 1000)
ax1.set(title="CO2 emitted per year", xlabel="year", ylabel="t CO2", yscale="log")
ax2.plot(result.gwi.dynamic / 1000, label="dynamic")
ax2.plot(result.gwi.static / 1000, label="static (no timing)", linestyle="--")
ax2.set(title="GWI, cumulative", xlabel="year", ylabel="t CO2e")
ax2.legend()
fig.tight_layout()

The static line jumps when each emission happens. The dynamic line rises
more slowly, because an emission's forcing builds up over the years after
it.

## How much the later emissions count

At a fixed horizon, a later emission has fewer years left to warm before
the horizon ends. The *sliding* GWI counts each emission only up to year
100. Compare it with the static GWI, where timing plays no part:

In [ ]:
sliding = result.gwi.sliding[-1]
static = result.gwi.static[-1]
print(f"static GWI:  {static:10,.0f} kg CO2e")
print(f"sliding GWI: {sliding:10,.0f} kg CO2e ({sliding / static:.0%} of static)")

The construction emissions dominate this frame, so the difference comes
from the smaller B4 and end-of-life amounts.

[`assess_building()`](../api/inventory.md#dlca.inventory.assess_building)
does both steps in one call and adds breakdowns; its total is the same
result:

In [ ]:
import numpy as np

from dlca import assess_building

building_result = assess_building(frame, parameters="AR6", horizon=100)
np.array_equal(building_result.total.gwi.dynamic, result.gwi.dynamic)

Next: [a whole building, broken down](03_building.ipynb).